In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2

import json
import time
from collections import defaultdict

import pandas as pd

from src.config import LLM_MODELS, RESULTS_DIR
from src.data_loader import load_churn, load_housing, load_bank
from src.llm_client import LLMClient
from src.prompts import (
    DATASET_CONFIG,
    build_zero_shot_prompt,
    build_with_stats_prompt,
)
from src.feature_engineer import apply_features

print("✅ Setup complete")

In [ ]:
datasets = {
    "churn": load_churn(verbose=False),
    "housing": load_housing(verbose=False),
    "bank": load_bank(verbose=False)
}

for name, (X, y) in datasets.items():
    print(f"   {name:8s}: {X.shape[0]} rows and {y.shape[0]} columns")

print("\n")
print(list(datasets["housing"][0].columns)[:8])

In [ ]:
prompt_variants = ["zero_shot", "with_stats"]
llm_keys = list(LLM_MODELS.keys())

jobs = []
for ds_name, (X, y) in datasets.items():
    cfg = DATASET_CONFIG[ds_name]
    for variant in prompt_variants:
        if variant == "zero_shot":
            prompt = build_zero_shot_prompt(
                dataset_name=cfg["display_name"],
                task=cfg["task"],
                target=cfg["target"],
                columns=list(X.columns),
            )
        else:
            prompt = build_with_stats_prompt(
                dataset_name=cfg["display_name"],
                task=cfg["task"],
                target=cfg["target"],
                df=X,
            )
        for llm_key in llm_keys:
            jobs.append({
                "dataset": ds_name,
                "llm": llm_key,
                "variant": variant,
                "prompt": prompt,
            })

print(f"📋 {len(jobs)} jobs queued")
print(f"   = {len(datasets)} datasets × {len(llm_keys)} LLMs × {len(prompt_variants)} prompts")

In [ ]:
# Re-run ONLY housing jobs with sanitized column names
housing_jobs = [j for j in jobs if j["dataset"] == "housing"]

print(f"Re-running {len(housing_jobs)} housing jobs...\n")
start = time.perf_counter()

# Initialize the dict so it works even if cell 4 wasn't run yet
all_suggestions = defaultdict(dict)

for i, job in enumerate(housing_jobs, 1):
    key = f"{job['llm']}__{job['variant']}"
print(f"[{i:2d}/{len(housing_jobs)}] {job['llm']:14s} | {job['variant']:10s}", end=" ")

try:
    llm = LLMClient(job["llm"])
    features = llm.suggest_features(
        prompt=job["prompt"],
        dataset_name="housing",
        prompt_variant=job["variant"],
    )
    all_suggestions["housing"][key] = features
    print(f"→ {len(features)} suggestions ✓")
except Exception as e:
    all_suggestions["housing"][key] = []
    print(f"→ ERROR: {type(e).__name__}: {str(e)[:80]} ✗")

print(f"\n⏱️  Housing re-run took {time.perf_counter() - start:.1f}s")

In [ ]:
all_suggestions = {ds: {} for ds in datasets}
all_metrics = []  # ← NEW: list of dicts for cost/latency tracking
errors = []

start = time.perf_counter()

for i, job in enumerate(jobs, 1):
    key = f"{job['llm']}__{job['variant']}"
    print(f"[{i:2d}/{len(jobs)}] {job['dataset']:8s} | {job['llm']:14s} | {job['variant']:10s}", end=" ")

    try:
        llm = LLMClient(job["llm"])
        features, metrics = llm.suggest_features(  # ← unpack tuple now
            prompt=job["prompt"],
            dataset_name=job["dataset"],
            prompt_variant=job["variant"],
        )
        all_suggestions[job["dataset"]][key] = features

        # Record metrics
        all_metrics.append({
            "dataset": job["dataset"],
            "llm": job["llm"],
            "variant": job["variant"],
            "n_features": len(features),
            **metrics.to_dict(),
        })

        print(
            f"→ {len(features)} feats | {metrics.total_tokens:>5} tok | ${metrics.cost_usd:.4f} | {metrics.latency_s:.1f}s ✓")
    except Exception as e:
        errors.append({**job, "error": str(e)})
        all_suggestions[job["dataset"]][key] = []
        all_metrics.append({
            "dataset": job["dataset"],
            "llm": job["llm"],
            "variant": job["variant"],
            "n_features": 0,
            "model_key": job["llm"],
            "model_id": LLM_MODELS[job["llm"]],
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
            "cost_usd": 0.0, "latency_s": 0.0,
            "success": False, "error": str(e),
        })
        print(f"→ ERROR: {type(e).__name__}: {str(e)[:80]} ✗")

elapsed = time.perf_counter() - start
total_cost = sum(m["cost_usd"] for m in all_metrics)
total_tokens = sum(m["total_tokens"] for m in all_metrics)
print(f"\n⏱️  Total time:   {elapsed:.1f}s ({elapsed / 60:.1f} min)")
print(f"💰 Total cost:    ${total_cost:.4f}")
print(f"🔢 Total tokens:  {total_tokens:,}")
print(f"❌ Errors:        {len(errors)}/{len(jobs)}")

In [ ]:
suggestions_path = RESULTS_DIR / "llm_suggestions.json"
with open(suggestions_path, "w") as f:
    json.dump(dict(all_suggestions), f, indent=2)

print(f"💾 Saved suggestions to {suggestions_path}")

if errors:
    errors_path = RESULTS_DIR / "llm_errors.json"
    with open(errors_path, "w") as f:
        # Strip out the long prompt text for readability
        clean_errors = [{k: v for k, v in e.items() if k != "prompt"} for e in errors]
        json.dump(clean_errors, f, indent=2)
    print(f"💾 Saved {len(errors)} errors to {errors_path}")

In [ ]:
validity_rows = []

for ds_name, (X, _) in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"📊 {ds_name.upper()}")
    print('=' * 70)

    for key, features in all_suggestions[ds_name].items():
        llm_name, variant = key.split("__")
        if not features:
            validity_rows.append({
                "dataset": ds_name, "llm": llm_name, "variant": variant,
                "n_suggested": 0, "n_applied": 0, "validity_rate": 0.0,
            })
            print(f"  {llm_name:14s} / {variant:12s}: SKIPPED (no suggestions)")
            continue

        _, report = apply_features(
            X, features,
            dataset_name=ds_name,
            llm=llm_name,
            prompt_variant=variant,
            verbose=False,
        )
        validity_rows.append({
            "dataset": ds_name,
            "llm": llm_name,
            "variant": variant,
            "n_suggested": report.n_suggested,
            "n_applied": report.n_applied,
            "validity_rate": report.validity_rate,
        })
        print(f"  {report.summary()}")

df_validity = pd.DataFrame(validity_rows)
df_validity.to_csv(RESULTS_DIR / "validity_rates.csv", index=False)
print(df_validity)

In [ ]:
# Save metrics CSV
df_metrics = pd.DataFrame(all_metrics)
metrics_path = RESULTS_DIR / "llm_metrics.csv"
df_metrics.to_csv(metrics_path, index=False)
print(f"💾 Saved {len(df_metrics)} call records to {metrics_path}\n")

# Per-LLM summary
summary = (
    df_metrics.groupby("llm")
    .agg(
        n_calls=("llm", "size"),
        total_tokens=("total_tokens", "sum"),
        total_cost=("cost_usd", "sum"),
        mean_latency=("latency_s", "mean"),
        mean_cost_per_call=("cost_usd", "mean"),
    )
    .round(4)
    .sort_values("total_cost", ascending=False)
)
summary

In [18]:
# Merge metrics with validity data to compute cost per valid feature
merged = df_metrics.merge(
    df_validity[["dataset", "llm", "variant", "n_applied"]],
    on=["dataset", "llm", "variant"],
    how="left",
)

# Cost-per-valid-feature analysis
cpv = (
    merged.groupby("llm")
    .agg(
        total_cost=("cost_usd", "sum"),
        total_valid_features=("n_applied", "sum"),
        total_suggested=("n_features", "sum"),
        mean_latency=("latency_s", "mean"),
    )
    .reset_index()
)
cpv["cost_per_valid_feature"] = cpv["total_cost"] / cpv["total_valid_features"].clip(lower=1)
cpv["validity_rate"] = cpv["total_valid_features"] / cpv["total_suggested"].clip(lower=1)
cpv = cpv.sort_values("cost_per_valid_feature")
cpv.round(6)

print(cpv)

             llm  total_cost  total_valid_features  total_suggested  \
2   gemini-flash    0.001732                    38               42   
4      llama-3.3    0.001644                    36               42   
3    gpt-4o-mini    0.002205                    39               42   
5       qwen-2.5    0.003121                    40               42   
1    deepseek-v3    0.004301                    36               42   
0  claude-sonnet    0.065379                    38               42   

   mean_latency  cost_per_valid_feature  validity_rate  
2      3.461220                0.000046       0.904762  
4     12.846827                0.000046       0.857143  
3      3.288990                0.000057       0.928571  
5     22.028256                0.000078       0.952381  
1     40.462070                0.000119       0.857143  
0      9.261426                0.001720       0.904762  


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURES_DIR

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Total cost per LLM
cpv_sorted = cpv.sort_values("total_cost", ascending=True)
axes[0].barh(cpv_sorted["llm"], cpv_sorted["total_cost"], color="#C44E52")
axes[0].set_title("Total cost per LLM ($USD)")
axes[0].set_xlabel("Cost")
for i, v in enumerate(cpv_sorted["total_cost"]):
    axes[0].text(v + 0.001, i, f"${v:.4f}", va="center", fontsize=9)

# 2. Mean latency per LLM
cpv_lat = cpv.sort_values("mean_latency", ascending=True)
axes[1].barh(cpv_lat["llm"], cpv_lat["mean_latency"], color="#4C72B0")
axes[1].set_title("Mean latency per call (seconds)")
axes[1].set_xlabel("Seconds")
for i, v in enumerate(cpv_lat["mean_latency"]):
    axes[1].text(v + 0.1, i, f"{v:.1f}s", va="center", fontsize=9)

# 3. Cost per valid feature (the killer chart!)
cpv_eff = cpv.sort_values("cost_per_valid_feature", ascending=True)
axes[2].barh(cpv_eff["llm"], cpv_eff["cost_per_valid_feature"], color="#55A868")
axes[2].set_title("Cost per VALID feature ($USD)")
axes[2].set_xlabel("Cost per valid feature")
for i, v in enumerate(cpv_eff["cost_per_valid_feature"]):
    axes[2].text(v + 0.0001, i, f"${v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_cost_latency_efficiency.png", bbox_inches="tight", dpi=120)
plt.show()

In [ ]:
print("Sample features per LLM (churn, zero-shot):\n")
for llm_key in llm_keys:
    key = f"{llm_key}__zero_shot"
    features = all_suggestions["churn"].get(key, [])
    print(f"\n{'─' * 70}")
    print(f"{llm_key}")
    print('─' * 70)
    for f in features[:3]:
        print(f"  • {f.get('name', '?'):35s} = {f.get('formula', '?')}")

In [ ]:
print(json.dumps(all_suggestions["housing"]["claude-sonnet__zero_shot"][:2], indent=2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURES_DIR

sns.set_style("whitegrid")

avg_per_llm = df_validity.groupby("llm")["validity_rate"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: avg validity per LLM
avg_per_llm.plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title("Average validity rate per LLM")
axes[0].set_xlabel("Validity rate")
axes[0].set_xlim(0, 1)
for i, v in enumerate(avg_per_llm):
    axes[0].text(v + 0.01, i, f"{v:.0%}", va="center")

# Right: heatmap of dataset × llm
pivot = df_validity.groupby(["llm", "dataset"])["validity_rate"].mean().unstack()
sns.heatmap(pivot, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1,
            ax=axes[1], cbar_kws={"label": "validity rate"})
axes[1].set_title("Validity rate by LLM × dataset")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_validity_rates.png", bbox_inches="tight")
plt.show()